### Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

- Run a hyperparameter sweep on a training script

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform


In [1]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

d:\mlflow_ML\mlflow_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
## Split the data into training,validation and test sets

train,test=train_test_split(data,test_size=0.25,random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [4]:
train[['quality']].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [5]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

## test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

## splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [6]:
np.mean(train_x,axis=0)

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [7]:
train_x.shape[1]

11

In [8]:
### ANN Model

def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    ## Define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)

        ]
    )

    ## compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params wwith MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),
                  epochs=epochs,
                  batch_size=64)
        
        ## Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        ## log the model

        mlflow.tensorflow.log_model(model,"model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [9]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [10]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [ ]:
# import mlflow
# import os

# print("Tracking URI:", mlflow.get_tracking_uri())
# print("MLFLOW_TRACKING_URI env:", os.getenv("MLFLOW_TRACKING_URI"))

Tracking URI: file:///d:/mlflow_ML/2-DLMLFLOW/mlruns
MLFLOW_TRACKING_URI env: None


In [14]:
# import mlflow

# exp = mlflow.get_experiment_by_name("wine-quality")
# print("Experiment ID:", exp.experiment_id)
# print("Artifact Location:", exp.artifact_location)

In [15]:
import os
os.makedirs(r"d:\mlflow_ML\2-DLMLFLOW\mlruns", exist_ok=True)

In [16]:
import mlflow

mlflow.set_tracking_uri("file:///d:/mlflow_ML/2-DLMLFLOW/mlruns")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:///d:/mlflow_ML/2-DLMLFLOW/mlruns


In [17]:
mlflow.set_experiment("wine-quality-v2")

2026/03/25 18:55:18 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality-v2' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///d:/mlflow_ML/2-DLMLFLOW/mlruns/999316357948089559', creation_time=1774445118544, experiment_id='999316357948089559', last_update_time=1774445118544, lifecycle_stage='active', name='wine-quality-v2', tags={}, workspace='default'>

In [18]:
exp = mlflow.get_experiment_by_name("wine-quality-v2")
print("Experiment ID:", exp.experiment_id)
print("Artifact Location:", exp.artifact_location)

Experiment ID: 999316357948089559
Artifact Location: file:///d:/mlflow_ML/2-DLMLFLOW/mlruns/999316357948089559


In [19]:
from hyperopt import fmin, tpe, Trials

with mlflow.start_run():
    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 21s 488ms/step - loss: 34.7531 - root_mean_squared_error: 5.8952
25/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.1635 - root_mean_squared_error: 5.1953   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 14.5490 - root_mean_squared_error: 3.8143 - val_loss: 4.2696 - val_root_mean_squared_error: 2.0663

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - loss: 3.5427 - root_mean_squared_error: 1.8822
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.4156 - root_mean_squared_error: 1.8475 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.9069 - root_mean_squared_error: 1.7050 - val_loss: 2.2387 - val_root_mean_squared_error: 1.4962

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 2.2850 - root_mean_squared_error: 1.5116
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1484 - root_mean_squared_error: 1.4643 
46

2026/03/25 18:56:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 20s 461ms/step - loss: 40.2090 - root_mean_squared_error: 6.3411
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13.8873 - root_mean_squared_error: 3.5996   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.4170 - root_mean_squared_error: 2.3274 - val_loss: 0.8958 - val_root_mean_squared_error: 0.9465

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 15s 350ms/step - loss: 0.9419 - root_mean_squared_error: 0.9705
27/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7751 - root_mean_squared_error: 0.8792   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6504 - root_mean_squared_error: 0.8065 - val_loss: 0.5695 - val_root_mean_squared_error: 0.7546

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.3364 - root_mean_squared_error: 0.5800
36/46 ━━━━

2026/03/25 18:56:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 22s 508ms/step - loss: 32.8283 - root_mean_squared_error: 5.7296
34/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17.6987 - root_mean_squared_error: 4.1250   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 7.8308 - root_mean_squared_error: 2.7984 - val_loss: 2.3873 - val_root_mean_squared_error: 1.5451

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - loss: 2.5774 - root_mean_squared_error: 1.6054
40/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.6776 - root_mean_squared_error: 1.2930 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4159 - root_mean_squared_error: 1.1899 - val_loss: 1.1056 - val_root_mean_squared_error: 1.0515

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - loss: 1.1268 - root_mean_squared_error: 1.0615
32/46 ━━━━━━━━

...

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 22s 491ms/step - loss: 35.4980 - root_mean_squared_error: 5.9580
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19.4627 - root_mean_squared_error: 4.3251   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 7.5253 - root_mean_squared_error: 2.7432 - val_loss: 1.9055 - val_root_mean_squared_error: 1.3804

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 14s 332ms/step - loss: 1.6483 - root_mean_squared_error: 1.2838
15/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.6877 - root_mean_squared_error: 1.2990   
38/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6898 - root_mean_squared_error: 1.2998
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.6116 - root_mean_squared_error: 1.2695 - val_loss: 1.4365 - val_root_mean_squared_error: 1.1985

Epoch 3/3                                                                      

 1/46 ━━━━━

2026/03/25 18:56:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:45<00:00, 11.42s/trial, best loss: 0.7384467124938965]

2026/03/25 18:56:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.0079728754098937), 'momentum': np.float64(0.9221223147979848)}
Best eval rmse: 0.7384467124938965


In [24]:
from hyperopt import fmin, tpe, Trials
import mlflow
import mlflow.pyfunc
import mlflow.tensorflow
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

with mlflow.start_run() as run:
    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])

    mlflow.tensorflow.log_model(
        model=best_run["model"],
        name="model",
        signature=signature
    )

    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

    # -----------------------------
    # Get run id and model uri
    # -----------------------------
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print("Run ID:", run_id)
    print("Model URI:", model_uri)

    # -----------------------------
    # Load as pyfunc
    # -----------------------------
    loaded_pyfunc_model = mlflow.pyfunc.load_model(model_uri)
    pyfunc_preds = loaded_pyfunc_model.predict(pd.DataFrame(test_x))
    pyfunc_rmse = np.sqrt(mean_squared_error(test_y, pyfunc_preds))

    print("PyFunc Predictions (first 5):")
    print(pyfunc_preds[:5])
    print("PyFunc Test RMSE:", pyfunc_rmse)

    # -----------------------------
    # Load as tensorflow
    # -----------------------------
    loaded_tf_model = mlflow.tensorflow.load_model(model_uri)
    tf_preds = loaded_tf_model.predict(test_x)
    tf_rmse = np.sqrt(mean_squared_error(test_y, tf_preds))

    print("TensorFlow Predictions (first 5):")
    print(tf_preds[:5])
    print("TensorFlow Test RMSE:", tf_rmse)

    # -----------------------------
    # Register model
    # -----------------------------
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name="wine-quality-model"
    )

    print("Registered Model Name:", registered_model.name)
    print("Registered Model Version:", registered_model.version)

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 19s 438ms/step - loss: 40.7842 - root_mean_squared_error: 6.3863
26/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21.3637 - root_mean_squared_error: 4.5192   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.7429 - root_mean_squared_error: 2.5967 - val_loss: 1.5006 - val_root_mean_squared_error: 1.2250

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 16s 362ms/step - loss: 1.4672 - root_mean_squared_error: 1.2113
17/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4276 - root_mean_squared_error: 1.1946   
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3655 - root_mean_squared_error: 1.1683
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.2772 - root_mean_squared_error: 1.1301 - val_loss: 1.1465 - val_root_mean_squared_error: 1.0707

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 1.1277 - root_mean_squared_error: 1.0619


2026/03/25 19:03:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 19s 439ms/step - loss: 35.2877 - root_mean_squared_error: 5.9403
27/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 29.3852 - root_mean_squared_error: 5.4103   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 18.6281 - root_mean_squared_error: 4.3160 - val_loss: 8.2190 - val_root_mean_squared_error: 2.8669

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 8.4426 - root_mean_squared_error: 2.9056
29/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.8418 - root_mean_squared_error: 2.6119 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.8497 - root_mean_squared_error: 2.2022 - val_loss: 2.9922 - val_root_mean_squared_error: 1.7298

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 2.6876 - root_mean_squared_error: 1.6394
32/46 ━━━━━━━

2026/03/25 19:03:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 18s 414ms/step - loss: 32.8829 - root_mean_squared_error: 5.7344
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.9757 - root_mean_squared_error: 5.2825   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 20.3831 - root_mean_squared_error: 4.5148 - val_loss: 10.6634 - val_root_mean_squared_error: 3.2655

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 9.5649 - root_mean_squared_error: 3.0927
34/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.9116 - root_mean_squared_error: 2.9815 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.7444 - root_mean_squared_error: 2.5970 - val_loss: 4.1129 - val_root_mean_squared_error: 2.0280

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 4.0107 - root_mean_squared_error: 2.0027
31/46 ━━━━━━

...

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
PyFunc Predictions (first 5):
[[5.83146  ]
 [6.949209 ]
 [6.299014 ]
 [5.3540697]
 [5.718166 ]]
PyFunc Test RMSE: 0.8698763701412627


39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
TensorFlow Predictions (first 5):
[[5.83146  ]
 [6.949209 ]
 [6.299014 ]
 [5.3540697]
 [5.718166 ]]
TensorFlow Test RMSE: 0.8698763701412627


d:\mlflow_ML\mlflow_env\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'wine-quality-model'.
2026/03/25 19:04:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 4df7af6ec2084e5bbdd4c6132dc1601f has no artifacts at artifact path 'model', registering model based on models:/m-432b0ea9f20749dda65a831ae9cc0282 instead


Registered Model Name: wine-quality-model
Registered Model Version: 1


Created version '1' of model 'wine-quality-model'.


In [20]:
# mlflow.set_experiment("wine-quality")
# with mlflow.start_run():
#     # Conduct the hyperparameter search using Hyperopt
#     trials=Trials()
#     best=fmin(
#         fn=objective,
#         space=space,
#         algo=tpe.suggest,
#         max_evals=4,
#         trials=trials
#     )

#     # Fetch the details of the best run
#     # best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

#     # Log the best parameters, loss, and model
#     mlflow.log_params(best)
#     mlflow.log_metric("eval_rmse", best_run["loss"])
#     mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

#     # Print out the best parameters and corresponding loss
#     print(f"Best parameters: {best}")
#     print(f"Best eval rmse: {best_run['loss']}")


In [22]:
# ## Inferencing

# from mlflow.models import validate_serving_input

# model_uri = 'runs:/9e0b52639ab44e6a864f5bd2f460fe42/model'

# # The logged model does not contain an input_example.
# # Manually generate a serving payload to verify your model prior to deployment.
# from mlflow.models import convert_input_example_to_serving_input

# # Define INPUT_EXAMPLE via assignment with your own input example to the model
# # A valid input example is a data instance suitable for pyfunc prediction
# serving_payload = convert_input_example_to_serving_input(test_x)

# # Validate the serving payload works on the model
# validate_serving_input(model_uri, serving_payload)

In [23]:
# # Load model as a PyFuncModel.
# model_uri = 'runs:/9e0b52639ab44e6a864f5bd2f460fe42/model'
# loaded_model = mlflow.pyfunc.load_model(model_uri)

# # Predict on a Pandas DataFrame.
# import pandas as pd
# loaded_model.predict(pd.DataFrame(test_x))

In [19]:
## Register in the model registry
mlflow.register_model(model_uri,"wine-quality")

Successfully registered model 'wine-quality'.
Created version '1' of model 'wine-quality'.


<ModelVersion: aliases=[], creation_timestamp=1727548664831, current_stage='None', description=None, last_updated_timestamp=1727548664831, name='wine-quality', run_id='9e0b52639ab44e6a864f5bd2f460fe42', run_link=None, source='file:///e:/UDemy%20Final/MlFlowStarter/2-DLMLFLOW/mlruns/929139454095764868/9e0b52639ab44e6a864f5bd2f460fe42/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>